In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from xgboost import XGBClassifier
import shap


# ============================================================
# 0. Configuration
# ============================================================
def find_project_root():
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists() and (path / "Result").exists():
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )

PROJECT_ROOT = find_project_root()

# Output directory from Script 01.
INPUT_ROOT = PROJECT_ROOT / "Result" / "01_Foldwise_Preprocessing_and_Ratio_Features"

# Output directory for this script.
OUT_ROOT = (
    PROJECT_ROOT
    / "Result"
    / "04_Foldwise_Correlation_Clustering_Champion_Selection_Rho_Sensitivity"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

TYPE_COL = "Type"

NON_NUM_COLS = {
    "No.",
    "Sample",
    "Type",
    "Reference"
}
IMPUTATION_METHODS = ["global_mean", "knn"]

# Number of outer folds. This must be consistent with Script 01.
N_OUTER_FOLDS = 5

# Inner CV is used only within each outer training fold.
N_INNER_SPLITS = 5

SEED = 42

# Rho-threshold list for sensitivity analysis.
RHO_LIST = [0.75, 0.80, 0.85, 0.90, 0.95]

# SHAP stability: Top-K features are selected in each inner fold.
TOPK = 50

# Ablation analysis is computationally expensive.
# It is recommended to run ablation only for rho = 0.90.
RUN_ABLATION_FOR_RHOS = [0.90]

# Feature-retention rule for selected champion features.
MIN_POSITIVE_FOLDS = 3
MIN_DELTA_F1 = 0.0

# Whether to save the large Spearman correlation matrix in each workbook.
SAVE_SPEARMAN_CORR_IN_WORKBOOK = False

# Whether to use GPU.
USE_GPU = True

XGB_PARAMS = dict(
    n_estimators=900,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.75,
    gamma=0.0,
    reg_lambda=6.0,
    reg_alpha=0.0,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
    device="cuda" if USE_GPU else "cpu",
)


# ============================================================
# 1. Basic utility functions
# ============================================================

def safe_tag_rho(rho):
    return f"{rho:.2f}"


def feature_complexity(name):
    s = str(name)
    ops = sum(s.count(ch) for ch in ["+", "-", "*", "/", "(", ")", "^"])
    return ops, len(s)


def is_ratio_feature(name):
    s = str(name)
    return s.startswith("R_Major_") or s.startswith("R_Trace_")


def build_model():
    return XGBClassifier(**XGB_PARAMS)


def get_feature_cols_from_train(train_df):
    """
    Determine usable feature columns only from the outer training set.

    Constant-column filtering must not use the test set; otherwise,
    test information would enter feature selection.
    """
    candidate_cols = [
        c for c in train_df.columns
        if c not in NON_NUM_COLS
    ]

    X_train = train_df[candidate_cols].copy()

    for c in X_train.columns:
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce")

    X_train = X_train.replace([np.inf, -np.inf], np.nan)

    # Remove columns with all NaN values.
    X_train = X_train.loc[:, X_train.notna().any(axis=0)]

    # Remove constant columns in the training set.
    X_train = X_train.loc[:, X_train.nunique(dropna=True) > 1]

    feature_cols = X_train.columns.tolist()

    return feature_cols


def prepare_train_test_xy(train_df, test_df):
    """
    Construct X_train, X_test, y_train, and y_test after reading the
    outer training and test datasets.

    Feature columns are determined only from the training set.
    """
    if TYPE_COL not in train_df.columns or TYPE_COL not in test_df.columns:
        raise ValueError(f"Label column was not found: {TYPE_COL}")

    feature_cols = get_feature_cols_from_train(train_df)

    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()

    for c in feature_cols:
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce")

    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)

    # In the formal workflow, Script 01 has already guaranteed that
    # no NaN or inf values remain. The checks below are retained here
    # as safeguards.
    if X_train.isna().sum().sum() > 0:
        raise ValueError(
            "NaN values remain in X_train. Please check the previous "
            "fold-wise preprocessing step."
        )

    if X_test.isna().sum().sum() > 0:
        raise ValueError(
            "NaN values remain in X_test. Please check the previous "
            "fold-wise preprocessing step."
        )

    y_train_raw = train_df[TYPE_COL].astype(str).reset_index(drop=True)
    y_test_raw = test_df[TYPE_COL].astype(str).reset_index(drop=True)

    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw.values)
    y_test = le.transform(y_test_raw.values)

    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)

    return X_train, X_test, y_train, y_test, y_train_raw, y_test_raw, le, feature_cols


def evaluate_model_on_test(model, X_test, y_test, label_encoder, tag):
    """
    Calculate multiclass test-set metrics.
    """
    pred = model.predict(X_test)

    metrics = {
        "Model": tag,
        "Accuracy": float(accuracy_score(y_test, pred)),
        "Balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "Macro_precision": float(
            precision_score(y_test, pred, average="macro", zero_division=0)
        ),
        "Macro_recall": float(
            recall_score(y_test, pred, average="macro", zero_division=0)
        ),
        "Macro_F1": float(
            f1_score(y_test, pred, average="macro", zero_division=0)
        ),
    }

    class_report_rows = []
    classes = list(range(len(label_encoder.classes_)))

    prec_cls = precision_score(
        y_test,
        pred,
        average=None,
        labels=classes,
        zero_division=0
    )

    rec_cls = recall_score(
        y_test,
        pred,
        average=None,
        labels=classes,
        zero_division=0
    )

    f1_cls = f1_score(
        y_test,
        pred,
        average=None,
        labels=classes,
        zero_division=0
    )

    for i, cls_name in enumerate(label_encoder.classes_):
        class_report_rows.append({
            "Model": tag,
            "Class": cls_name,
            "Precision": float(prec_cls[i]),
            "Recall": float(rec_cls[i]),
            "F1": float(f1_cls[i])
        })

    cm = confusion_matrix(y_test, pred, labels=classes)
    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{c}" for c in label_encoder.classes_],
        columns=[f"Pred_{c}" for c in label_encoder.classes_]
    )

    return metrics, pd.DataFrame(class_report_rows), cm_df


def train_and_evaluate_feature_set(
    X_train,
    y_train,
    X_test,
    y_test,
    features,
    label_encoder,
    tag
):
    """
    Train a model on the outer training set using a given feature set
    and evaluate it on the outer test set.
    """
    if len(features) == 0:
        raise ValueError(f"The feature set for {tag} is empty.")

    model = build_model()
    model.fit(X_train[features], y_train)

    metrics, class_df, cm_df = evaluate_model_on_test(
        model,
        X_test[features],
        y_test,
        label_encoder,
        tag
    )

    metrics["N_features"] = len(features)

    return metrics, class_df, cm_df


# ============================================================
# 2. Union-Find correlation clustering
# ============================================================

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


def compute_spearman_corr(X):
    """
    Compute the Spearman correlation matrix only on the outer training set.
    """
    corr = X.corr(method="spearman")
    return corr


def spearman_clusters_from_corr(corr, rho_th):
    """
    Perform correlation clustering based on the training-fold Spearman
    matrix and the threshold rho_th.
    """
    n = corr.shape[0]
    cols = list(corr.columns)
    arr = corr.values

    uf = UnionFind(n)

    for i in range(n):
        row = arr[i, i + 1:]
        idx = np.where(np.abs(row) >= rho_th)[0]

        for off in idx:
            j = i + 1 + int(off)
            uf.union(i, j)

    root_to_feats = {}

    for i in range(n):
        r = uf.find(i)
        root_to_feats.setdefault(r, []).append(cols[i])

    clusters = list(root_to_feats.values())

    return clusters


def plot_cluster_sizes(clusters, rho_th, out_path, title_prefix):
    sizes = [len(c) for c in clusters]

    plt.figure(figsize=(8, 5), dpi=300)
    plt.hist(sizes, bins=30)

    plt.title(
        f"{title_prefix} Cluster Size Distribution (|rho| >= {rho_th:.2f})",
        fontsize=14,
        fontweight="bold"
    )

    plt.xlabel("Cluster Size", fontsize=12, fontweight="bold")
    plt.ylabel("Count", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()


# ============================================================
# 3. Inner-CV SHAP importance and stability
# ============================================================

def shap_mean_abs_multiclass(model, X_val):
    """
    Return mean absolute SHAP values for each feature, aggregated across
    samples and classes.
    """
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_val)

    if isinstance(sv, list):
        sv_arr = np.stack([np.asarray(a) for a in sv], axis=2)
    else:
        sv_arr = np.asarray(sv)

        if sv_arr.ndim == 2:
            sv_arr = sv_arr[:, :, None]
        elif sv_arr.ndim == 3:
            pass
        else:
            raise ValueError(f"Unexpected SHAP shape: {sv_arr.shape}")

    mean_abs = np.mean(np.abs(sv_arr), axis=(0, 2))

    return mean_abs


def compute_inner_cv_shap_importance(X_train, y_train):
    """
    Perform inner CV within the outer training set.

    Outputs:
    - importance_mean
    - topk_freq_ratio
    - inner_full_f1_by_fold
    - inner_splits
    """
    inner_cv = StratifiedKFold(
        n_splits=N_INNER_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    inner_splits = list(inner_cv.split(X_train, y_train))

    F = X_train.shape[1]
    imp_sum = np.zeros(F, dtype=float)
    topk_cnt = np.zeros(F, dtype=int)
    inner_full_f1_by_fold = []

    for inner_fold, (tr_idx, va_idx) in enumerate(inner_splits, start=1):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]

        model = build_model()
        model.fit(X_tr, y_tr)

        pred = model.predict(X_va)
        f1m = f1_score(y_va, pred, average="macro", zero_division=0)
        inner_full_f1_by_fold.append(float(f1m))

        mean_abs = shap_mean_abs_multiclass(model, X_va)

        imp_sum += mean_abs

        k = min(TOPK, F)
        top_idx = np.argsort(-mean_abs)[:k]
        topk_cnt[top_idx] += 1

    importance_mean = imp_sum / N_INNER_SPLITS
    topk_freq_ratio = topk_cnt / N_INNER_SPLITS

    return importance_mean, topk_freq_ratio, inner_full_f1_by_fold, inner_splits


# ============================================================
# 4. Cluster-champion selection
# ============================================================

def select_cluster_champions(
    clusters,
    feature_names,
    importance_mean,
    topk_freq_ratio,
    missing_rate,
    zero_rate
):
    feat2idx = {f: i for i, f in enumerate(feature_names)}

    rows = []
    champions = []

    for cid, feats in enumerate(clusters, start=1):
        feats = [f for f in feats if f in feat2idx]

        if len(feats) == 0:
            continue

        cand = []

        for f in feats:
            i = feat2idx[f]

            imp = float(importance_mean[i])
            stab = float(topk_freq_ratio[i])
            score = imp * stab

            cand.append({
                "feature": f,
                "importance_mean": imp,
                "topk_freq_ratio": stab,
                "score": score,
                "missing_rate": float(missing_rate.get(f, 0.0)),
                "zero_rate": float(zero_rate.get(f, 0.0)),
                "complex_ops": feature_complexity(f)[0],
                "name_len": feature_complexity(f)[1],
                "is_ratio": is_ratio_feature(f),
            })

        cand_sorted = sorted(
            cand,
            key=lambda d: (
                -d["score"],
                d["missing_rate"],
                d["zero_rate"],
                d["complex_ops"],
                d["name_len"],
            )
        )

        champ = cand_sorted[0]
        champions.append(champ["feature"])

        rows.append({
            "cluster_id": cid,
            "cluster_size": len(feats),
            "champion": champ["feature"],
            "champion_score": champ["score"],
            "champion_importance_mean": champ["importance_mean"],
            "champion_topk_freq_ratio": champ["topk_freq_ratio"],
            "champion_missing_rate": champ["missing_rate"],
            "champion_zero_rate": champ["zero_rate"],
            "champion_is_ratio": champ["is_ratio"],
            "champion_complex_ops": champ["complex_ops"],
            "champion_name_len": champ["name_len"],
        })

    champ_df = (
        pd.DataFrame(rows)
        .sort_values("champion_score", ascending=False)
        .reset_index(drop=True)
    )

    return champions, champ_df


def make_feature_score_table(
    feature_names,
    importance_mean,
    topk_freq_ratio,
    missing_rate,
    zero_rate
):
    feat_score_df = pd.DataFrame({
        "feature": feature_names,
        "importance_mean_shap": importance_mean,
        "topk_freq_ratio": topk_freq_ratio,
        "score_importance_times_stability": importance_mean * topk_freq_ratio,
        "missing_rate": [float(missing_rate.get(f, 0.0)) for f in feature_names],
        "zero_rate": [float(zero_rate.get(f, 0.0)) for f in feature_names],
        "is_ratio": [is_ratio_feature(f) for f in feature_names],
        "complex_ops": [feature_complexity(f)[0] for f in feature_names],
        "name_len": [feature_complexity(f)[1] for f in feature_names],
    }).sort_values(
        "score_importance_times_stability",
        ascending=False
    ).reset_index(drop=True)

    return feat_score_df


def cv_f1_on_feature_subset(X_train_sub, y_train, inner_splits):
    """
    Evaluate a feature subset using inner CV within the outer training set.
    """
    f1s = []

    for tr_idx, va_idx in inner_splits:
        model = build_model()
        model.fit(X_train_sub.iloc[tr_idx], y_train[tr_idx])
        pred = model.predict(X_train_sub.iloc[va_idx])
        f1m = float(
            f1_score(
                y_train[va_idx],
                pred,
                average="macro",
                zero_division=0
            )
        )
        f1s.append(f1m)

    return float(np.mean(f1s)), float(np.std(f1s)), f1s


# ============================================================
# 5. Inner-CV ablation analysis
# ============================================================

def ablation_all_champions(
    X_train,
    y_train,
    inner_splits,
    champions,
    inner_full_f1_by_fold
):
    rows = []

    full_mean = float(np.mean(inner_full_f1_by_fold))
    full_std = float(np.std(inner_full_f1_by_fold))

    for j, feat in enumerate(champions, start=1):
        X_drop = X_train.drop(columns=[feat])

        f1_drop = []
        deltas = []

        for fold_id, (tr_idx, va_idx) in enumerate(inner_splits, start=1):
            model = build_model()
            model.fit(X_drop.iloc[tr_idx], y_train[tr_idx])
            pred = model.predict(X_drop.iloc[va_idx])

            f1m = float(
                f1_score(
                    y_train[va_idx],
                    pred,
                    average="macro",
                    zero_division=0
                )
            )

            f1_drop.append(f1m)
            deltas.append(float(inner_full_f1_by_fold[fold_id - 1] - f1m))

        deltas_arr = np.array(deltas, dtype=float)

        rows.append({
            "feature": feat,
            "inner_full_f1_mean": full_mean,
            "inner_full_f1_std": full_std,
            "inner_f1_drop_mean": float(np.mean(f1_drop)),
            "inner_f1_drop_std": float(np.std(f1_drop)),
            "delta_f1_mean_full_minus_drop": float(np.mean(deltas_arr)),
            "delta_f1_std": float(np.std(deltas_arr)),
            "positive_folds": int(np.sum(deltas_arr > 0)),
            "delta_f1_fold_values": ",".join([f"{v:.6f}" for v in deltas])
        })

        print(
            f"      Ablation {j}/{len(champions)}: {feat}, "
            f"Delta F1={np.mean(deltas_arr):+.4f}, "
            f"positive={np.sum(deltas_arr > 0)}/{N_INNER_SPLITS}",
            flush=True
        )

    ab_df = (
        pd.DataFrame(rows)
        .sort_values("delta_f1_mean_full_minus_drop", ascending=False)
        .reset_index(drop=True)
    )

    return ab_df


def select_features_from_ablation(ab_df):
    """
    Select more stable champion features based on inner-CV ablation results.
    """
    if ab_df.empty:
        return []

    selected = ab_df[
        (ab_df["positive_folds"] >= MIN_POSITIVE_FOLDS)
        &
        (ab_df["delta_f1_mean_full_minus_drop"] > MIN_DELTA_F1)
    ]["feature"].tolist()

    return selected


# ============================================================
# 6. Plotting functions
# ============================================================

def plot_ablation_hist(ab_df, out_path, rho_th, title_prefix):
    if ab_df.empty:
        return

    plt.figure(figsize=(8, 5), dpi=300)
    plt.hist(ab_df["delta_f1_mean_full_minus_drop"].values, bins=30)

    plt.title(
        f"{title_prefix} Ablation Delta Macro-F1 (|rho| >= {rho_th:.2f})",
        fontsize=14,
        fontweight="bold",
        pad=10
    )

    plt.xlabel("Delta Macro-F1 (Full - Drop Feature)", fontsize=12, fontweight="bold")
    plt.ylabel("Count", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()


def plot_ablation_top_bottom(ab_df, out_path, rho_th, title_prefix, k=25):
    if ab_df.empty:
        return

    top = ab_df.head(k).copy()
    bottom = ab_df.tail(k).copy()
    comb = pd.concat([top, bottom], axis=0)

    comb = comb.sort_values("delta_f1_mean_full_minus_drop", ascending=True)

    plt.figure(figsize=(12, 10), dpi=300)
    plt.barh(
        comb["feature"].astype(str),
        comb["delta_f1_mean_full_minus_drop"].values
    )

    plt.axvline(0, linewidth=1)

    plt.title(
        f"{title_prefix} Ablation on Champions (|rho| >= {rho_th:.2f})",
        fontsize=16,
        fontweight="bold",
        pad=12
    )

    plt.xlabel("Delta Macro-F1 (Full - Drop Feature)", fontsize=12, fontweight="bold")
    plt.ylabel("Cluster Champions", fontsize=12, fontweight="bold")
    plt.tick_params(axis="both", labelsize=8)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()


def plot_rho_sensitivity_one_method_fold(sens_df, out_path, title_prefix):
    if sens_df.empty:
        return

    rho = sens_df["rho_th"].values

    plt.figure(figsize=(10, 6), dpi=300)

    plt.plot(
        rho,
        sens_df["outer_full_macro_f1"].values,
        marker="s",
        label="Full-feature outer-test Macro-F1"
    )

    plt.plot(
        rho,
        sens_df["outer_champion_macro_f1"].values,
        marker="o",
        label="Champion-feature outer-test Macro-F1"
    )

    if "outer_selected_macro_f1" in sens_df.columns:
        plt.plot(
            rho,
            sens_df["outer_selected_macro_f1"].values,
            marker="^",
            label="Ablation-selected outer-test Macro-F1"
        )

    plt.xlabel("Spearman threshold |rho|", fontsize=12, fontweight="bold")
    plt.ylabel("Outer-test Macro-F1", fontsize=12, fontweight="bold")

    plt.title(
        f"{title_prefix} Rho Sensitivity",
        fontsize=14,
        fontweight="bold"
    )

    plt.grid(True, linewidth=0.3)
    plt.legend()

    for x, y, n in zip(
        rho,
        sens_df["outer_champion_macro_f1"].values,
        sens_df["n_champions"].values
    ):
        plt.text(x, y, f" n={int(n)}", fontsize=9)

    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()


# ============================================================
# 7. Main workflow for one method and one outer fold
# ============================================================

def process_one_outer_fold(method, outer_fold, train_path, test_path, out_dir):
    print(f"\n========== {method} outer fold {outer_fold} ==========")

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    train_df.columns = [str(c).strip() for c in train_df.columns]
    test_df.columns = [str(c).strip() for c in test_df.columns]

    X_train, X_test, y_train, y_test, y_train_raw, y_test_raw, le, feature_cols = prepare_train_test_xy(
        train_df,
        test_df
    )

    feature_names = list(X_train.columns)

    print(f"  Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
    print(f"  Features: {X_train.shape[1]}")
    print(f"  Classes: {le.classes_.tolist()}")

    # Missing rate, zero-value rate, and Spearman correlation are calculated
    # only on the outer training set.
    missing_rate = X_train.isna().mean(axis=0)
    zero_rate = (X_train == 0).mean(axis=0)

    corr = compute_spearman_corr(X_train)

    # Inner-CV SHAP is performed only within the outer training set.
    print("  Computing inner-CV SHAP importance...")
    importance_mean, topk_freq_ratio, inner_full_f1_by_fold, inner_splits = compute_inner_cv_shap_importance(
        X_train,
        y_train
    )

    inner_full_mean = float(np.mean(inner_full_f1_by_fold))
    inner_full_std = float(np.std(inner_full_f1_by_fold))

    feat_score_df = make_feature_score_table(
        feature_names,
        importance_mean,
        topk_freq_ratio,
        missing_rate,
        zero_rate
    )

    # Outer full-feature baseline.
    full_metrics, full_class_df, full_cm_df = train_and_evaluate_feature_set(
        X_train,
        y_train,
        X_test,
        y_test,
        feature_names,
        le,
        tag="Full_features"
    )

    sens_rows = []
    all_outer_metrics = []
    all_class_reports = []
    all_confusions = {}

    all_outer_metrics.append(full_metrics)
    all_class_reports.append(full_class_df)
    all_confusions["Full_features"] = full_cm_df

    for rho_th in RHO_LIST:
        tag = safe_tag_rho(rho_th)
        rho_dir = out_dir / f"rho_{tag}"
        rho_dir.mkdir(parents=True, exist_ok=True)

        print(f"  ---- rho={tag} ----")

        clusters = spearman_clusters_from_corr(corr, rho_th)

        sizes = [len(c) for c in clusters]

        cluster_size_plot = (
            rho_dir
            / f"outer_fold_{outer_fold:02d}_cluster_size_distribution_rho{tag}.png"
        )

        plot_cluster_sizes(
            clusters,
            rho_th,
            cluster_size_plot,
            title_prefix=f"{method} Fold {outer_fold}"
        )

        champions, champ_df = select_cluster_champions(
            clusters,
            feature_names,
            importance_mean,
            topk_freq_ratio,
            missing_rate,
            zero_rate
        )

        # Inner-CV champion-only performance.
        X_champ_train = X_train[champions].copy()
        inner_champ_mean, inner_champ_std, inner_champ_by_fold = cv_f1_on_feature_subset(
            X_champ_train,
            y_train,
            inner_splits
        )

        # Outer-test champion-only performance.
        champion_metrics, champion_class_df, champion_cm_df = train_and_evaluate_feature_set(
            X_train,
            y_train,
            X_test,
            y_test,
            champions,
            le,
            tag=f"Champions_rho_{tag}"
        )

        all_outer_metrics.append(champion_metrics)
        all_class_reports.append(champion_class_df)
        all_confusions[f"Champions_rho_{tag}"] = champion_cm_df

        # Cluster membership.
        cluster_rows = []
        for i, feats in enumerate(clusters, start=1):
            for f in feats:
                cluster_rows.append({
                    "cluster_id": i,
                    "feature": f
                })

        clusters_df = pd.DataFrame(cluster_rows)

        ab_df = pd.DataFrame()
        selected_features = []

        # Ablation analysis is recommended only for rho = 0.90.
        if any(abs(rho_th - r) < 1e-9 for r in RUN_ABLATION_FOR_RHOS):
            print(f"  Running ablation for rho={tag}...")

            ab_df = ablation_all_champions(
                X_train,
                y_train,
                inner_splits,
                champions,
                inner_full_f1_by_fold
            )

            selected_features = select_features_from_ablation(ab_df)

            # If no feature satisfies the ablation-selection rule,
            # fall back to the full champion set.
            if len(selected_features) == 0:
                selected_features = champions.copy()

            selected_metrics, selected_class_df, selected_cm_df = train_and_evaluate_feature_set(
                X_train,
                y_train,
                X_test,
                y_test,
                selected_features,
                le,
                tag=f"Selected_after_ablation_rho_{tag}"
            )

            all_outer_metrics.append(selected_metrics)
            all_class_reports.append(selected_class_df)
            all_confusions[f"Selected_after_ablation_rho_{tag}"] = selected_cm_df

            ab_hist = (
                rho_dir
                / f"outer_fold_{outer_fold:02d}_ablation_delta_hist_rho{tag}.png"
            )

            ab_top_bottom = (
                rho_dir
                / f"outer_fold_{outer_fold:02d}_ablation_top_bottom_rho{tag}.png"
            )

            plot_ablation_hist(
                ab_df,
                ab_hist,
                rho_th,
                title_prefix=f"{method} Fold {outer_fold}"
            )

            plot_ablation_top_bottom(
                ab_df,
                ab_top_bottom,
                rho_th,
                title_prefix=f"{method} Fold {outer_fold}",
                k=25
            )

            outer_selected_f1 = selected_metrics["Macro_F1"]
            n_selected = len(selected_features)

        else:
            selected_metrics = {}
            outer_selected_f1 = np.nan
            n_selected = np.nan

        # Save results for the current rho threshold.
        out_xlsx = (
            rho_dir
            / f"outer_fold_{outer_fold:02d}_cluster_champions_SHAP_innerCV_rho{tag}.xlsx"
        )

        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            feat_score_df.to_excel(
                writer,
                index=False,
                sheet_name="FeatureScores_SHAP_innerCV"
            )

            champ_df.to_excel(
                writer,
                index=False,
                sheet_name="ClusterChampions"
            )

            clusters_df.to_excel(
                writer,
                index=False,
                sheet_name="ClusterMembership"
            )

            if not ab_df.empty:
                ab_df.to_excel(
                    writer,
                    index=False,
                    sheet_name="Ablation_Champions_innerCV"
                )
            else:
                pd.DataFrame({
                    "Message": [f"Ablation not run for rho={tag}."]
                }).to_excel(
                    writer,
                    index=False,
                    sheet_name="Ablation_Champions_innerCV"
                )

            pd.DataFrame({
                "Metric": [
                    "inner_full_f1_mean",
                    "inner_full_f1_std",
                    "inner_champion_f1_mean",
                    "inner_champion_f1_std",
                    "outer_full_macro_f1",
                    "outer_champion_macro_f1",
                    "outer_selected_macro_f1"
                ],
                "Value": [
                    inner_full_mean,
                    inner_full_std,
                    inner_champ_mean,
                    inner_champ_std,
                    full_metrics["Macro_F1"],
                    champion_metrics["Macro_F1"],
                    outer_selected_f1
                ]
            }).to_excel(
                writer,
                index=False,
                sheet_name="PerformanceSummary"
            )

            pd.DataFrame({
                "selected_features_after_ablation": selected_features
            }).to_excel(
                writer,
                index=False,
                sheet_name="SelectedFeatures"
            )

            if SAVE_SPEARMAN_CORR_IN_WORKBOOK:
                corr.to_excel(writer, sheet_name="SpearmanCorr_trainOnly")

        sens_rows.append({
            "Method": method,
            "Outer_fold": outer_fold,
            "rho_th": float(rho_th),
            "n_features_total": int(X_train.shape[1]),
            "n_clusters": int(len(clusters)),
            "n_champions": int(len(champions)),
            "n_selected_after_ablation": n_selected,
            "reduction_ratio_champions_over_total": float(len(champions) / X_train.shape[1]),
            "cluster_size_min": int(np.min(sizes)),
            "cluster_size_median": float(np.median(sizes)),
            "cluster_size_max": int(np.max(sizes)),
            "inner_full_f1_mean": inner_full_mean,
            "inner_full_f1_std": inner_full_std,
            "inner_champion_f1_mean": inner_champ_mean,
            "inner_champion_f1_std": inner_champ_std,
            "outer_full_macro_f1": full_metrics["Macro_F1"],
            "outer_champion_macro_f1": champion_metrics["Macro_F1"],
            "outer_selected_macro_f1": outer_selected_f1,
            "cluster_size_plot": str(cluster_size_plot),
            "result_file": str(out_xlsx)
        })

        print(
            f"    clusters={len(clusters)}, champions={len(champions)}, "
            f"outer champion Macro-F1={champion_metrics['Macro_F1']:.4f}"
        )

    sens_df = pd.DataFrame(sens_rows)

    sens_path = (
        out_dir
        / f"outer_fold_{outer_fold:02d}_rho_sensitivity_summary.xlsx"
    )

    sens_df.to_excel(sens_path, index=False)

    rho_plot_path = (
        out_dir
        / f"outer_fold_{outer_fold:02d}_rho_sensitivity_plot.png"
    )

    plot_rho_sensitivity_one_method_fold(
        sens_df,
        rho_plot_path,
        title_prefix=f"{method} Outer Fold {outer_fold}"
    )

    outer_metrics_df = pd.DataFrame(all_outer_metrics)
    class_report_df = pd.concat(all_class_reports, axis=0, ignore_index=True)

    metrics_path = (
        out_dir
        / f"outer_fold_{outer_fold:02d}_outer_test_metrics.xlsx"
    )

    with pd.ExcelWriter(metrics_path, engine="openpyxl") as writer:
        outer_metrics_df.to_excel(
            writer,
            index=False,
            sheet_name="OuterTestMetrics"
        )

        class_report_df.to_excel(
            writer,
            index=False,
            sheet_name="ClasswiseMetrics"
        )

        for name, cm_df in all_confusions.items():
            safe_name = name[:31]
            cm_df.to_excel(
                writer,
                sheet_name=safe_name
            )

    print(f"  Saved fold summary: {sens_path}")
    print(f"  Saved outer metrics: {metrics_path}")

    return sens_df, outer_metrics_df, class_report_df


# ============================================================
# 8. Summary across outer folds
# ============================================================

def summarize_across_outer_folds(all_sens_df):
    """
    Summarize outer-test performance across methods and rho thresholds.
    """
    metric_cols = [
        "n_clusters",
        "n_champions",
        "n_selected_after_ablation",
        "reduction_ratio_champions_over_total",
        "inner_full_f1_mean",
        "inner_champion_f1_mean",
        "outer_full_macro_f1",
        "outer_champion_macro_f1",
        "outer_selected_macro_f1"
    ]

    agg_dict = {}

    for col in metric_cols:
        agg_dict[f"{col}_mean"] = (col, "mean")
        agg_dict[f"{col}_std"] = (col, "std")

    summary = (
        all_sens_df
        .groupby(["Method", "rho_th"], as_index=False)
        .agg(**agg_dict)
    )

    return summary


# ============================================================
# 9. Main function
# ============================================================

def main():
    all_sens_list = []
    all_outer_metrics_list = []
    all_class_reports_list = []

    for method in IMPUTATION_METHODS:
        method_input_dir = INPUT_ROOT / method

        if not method_input_dir.exists():
            raise FileNotFoundError(f"Input directory was not found: {method_input_dir}")

        method_out_dir = OUT_ROOT / method
        method_out_dir.mkdir(parents=True, exist_ok=True)

        for outer_fold in range(1, N_OUTER_FOLDS + 1):
            train_path = (
                method_input_dir
                / f"fold_{outer_fold:02d}_train_with_ratios.xlsx"
            )

            test_path = (
                method_input_dir
                / f"fold_{outer_fold:02d}_test_with_ratios.xlsx"
            )

            if not train_path.exists():
                raise FileNotFoundError(f"Training-fold file was not found: {train_path}")

            if not test_path.exists():
                raise FileNotFoundError(f"Test-fold file was not found: {test_path}")

            fold_out_dir = (
                method_out_dir
                / f"outer_fold_{outer_fold:02d}"
            )

            fold_out_dir.mkdir(parents=True, exist_ok=True)

            sens_df, outer_metrics_df, class_report_df = process_one_outer_fold(
                method,
                outer_fold,
                train_path,
                test_path,
                fold_out_dir
            )

            all_sens_list.append(sens_df)

            outer_metrics_df["Method"] = method
            outer_metrics_df["Outer_fold"] = outer_fold
            all_outer_metrics_list.append(outer_metrics_df)

            class_report_df["Method"] = method
            class_report_df["Outer_fold"] = outer_fold
            all_class_reports_list.append(class_report_df)

    all_sens_df = pd.concat(all_sens_list, axis=0, ignore_index=True)
    all_outer_metrics_df = pd.concat(all_outer_metrics_list, axis=0, ignore_index=True)
    all_class_reports_df = pd.concat(all_class_reports_list, axis=0, ignore_index=True)

    all_sens_path = (
        OUT_ROOT
        / "all_outer_folds_rho_sensitivity_raw.xlsx"
    )

    all_outer_metrics_path = (
        OUT_ROOT
        / "all_outer_folds_outer_test_metrics_raw.xlsx"
    )

    all_class_report_path = (
        OUT_ROOT
        / "all_outer_folds_classwise_metrics_raw.xlsx"
    )

    all_sens_df.to_excel(all_sens_path, index=False)
    all_outer_metrics_df.to_excel(all_outer_metrics_path, index=False)
    all_class_reports_df.to_excel(all_class_report_path, index=False)

    summary_df = summarize_across_outer_folds(all_sens_df)

    summary_path = (
        OUT_ROOT
        / "summary_across_outer_folds_by_method_and_rho.xlsx"
    )

    summary_df.to_excel(summary_path, index=False)

    print("\nAll tasks completed.")
    print(f"Output directory: {OUT_ROOT}")
    print(f"All outer-fold raw rho-sensitivity results: {all_sens_path}")
    print(f"All outer-fold test metrics: {all_outer_metrics_path}")
    print(f"Summary across outer folds: {summary_path}")

    print("\nConcise summary across outer folds:")
    show_cols = [
        "Method",
        "rho_th",
        "n_champions_mean",
        "outer_full_macro_f1_mean",
        "outer_champion_macro_f1_mean",
        "outer_selected_macro_f1_mean"
    ]

    existing = [c for c in show_cols if c in summary_df.columns]
    print(summary_df[existing])


# ============================================================
# 10. Program entry point
# ============================================================

if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. Path settings
# ============================================================

def find_project_root():
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists() and (path / "Result").exists():
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )

PROJECT_ROOT = find_project_root()

ROOT_DIR = (
    PROJECT_ROOT
    / "Result"
    / "04_Foldwise_Correlation_Clustering_Champion_Selection_Rho_Sensitivity"
)

OUT_DIR = ROOT_DIR / "Cluster_Champion_Stability_Figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_DIR = ROOT_DIR

RHO_TARGET = 0.90
TOTAL_FEATURES = 443

# ============================================================
# 1. Use existing result directory
# ============================================================

print("Result directory:", EXTRACT_DIR)


# ============================================================
# 2. Automatically find rho=0.90 cluster-champion files
# ============================================================

champion_files = list(
    EXTRACT_DIR.rglob(
        f"rho_{RHO_TARGET:.2f}/*cluster_champions_SHAP_innerCV_rho{RHO_TARGET:.2f}.xlsx"
    )
)

# Compatibility fallback for equivalent rho=0.90 naming.
if len(champion_files) == 0:
    champion_files = list(
        EXTRACT_DIR.rglob(
            "rho_0.90/*cluster_champions_SHAP_innerCV_rho0.90.xlsx"
        )
    )

if len(champion_files) == 0:
    raise FileNotFoundError(
        "No rho=0.90 cluster-champion files were found. "
        "Please check ROOT_DIR and the folder structure."
    )

print(f"Found {len(champion_files)} rho=0.90 cluster-champion files.")


# ============================================================
# 3. Feature-name cleaning functions
# ============================================================

def clean_feature_name(name):
    """
    Clean raw code-style feature names into manuscript-style labels.
    """
    s = str(name)

    s = s.replace("R_Major_", "")
    s = s.replace("R_Trace_", "")

    s = s.replace("10000*Ga/Al", "10000xGa/Al")
    s = s.replace("10000×Ga/Al", "10000xGa/Al")

    s = s.replace("SiO2(wt%)", "SiO2")

    return s


def get_method_and_fold(path):
    """
    Identify the imputation method and outer-fold ID from the file path.
    """
    low = str(path).lower().replace("\\", "/")

    if "/knn/" in low:
        method = "KNN"
    elif "/global_mean/" in low:
        method = "global-mean"
    else:
        method = "unknown"

    m = re.search(r"outer_fold_(\d+)", low)
    fold = int(m.group(1)) if m else np.nan

    return method, fold


def classify_feature(feature):
    """
    Add a simple feature-category label for manuscript tables.

    This label is used only for display and is not involved in any calculation.
    """
    f = str(feature)

    classical = {
        "Zr+Nb+Ce+Y": "Classical A-type / HFSE indicator",
        "10000xGa/Al": "Classical A-type indicator",
        "A/CNK": "Classical aluminosity indicator",
        "A/NK": "Classical aluminosity indicator",
    }

    if f in classical:
        return classical[f]

    if f == "L.O.I":
        return "Volatile-related variable"

    if f == "Total":
        return "Major-element total"

    hfse_ratio = {"Nb/Ta", "Zr/Hf", "Ta/Th", "Nb/Pb"}
    if f in hfse_ratio:
        return "HFSE-related ratio"

    ree_elements = [
        "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb",
        "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Y"
    ]

    major_elements = [
        "SiO2", "TiO2", "Al2O3", "Fe2O3t", "MgO",
        "CaO", "Na2O", "K2O", "MnO", "P2O5"
    ]

    if "/" in f:
        num, den = f.split("/", 1)

        if num in major_elements or den in major_elements:
            return "Major-element ratio"

        if num in ree_elements or den in ree_elements:
            return "REE-related ratio"

        return "Trace-element ratio"

    return "Elemental or composite variable"


# ============================================================
# 4. Read all rho=0.90 ClusterChampions sheets
# ============================================================

records = []

for fp in champion_files:
    method, fold = get_method_and_fold(fp)

    df = pd.read_excel(fp, sheet_name="ClusterChampions")

    required_cols = [
        "cluster_id",
        "cluster_size",
        "champion",
        "champion_score",
        "champion_importance_mean",
        "champion_topk_freq_ratio"
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"The following columns are missing in {fp}: {missing}")

    for _, row in df.iterrows():
        feature_clean = clean_feature_name(row["champion"])

        records.append({
            "Method": method,
            "Fold": fold,
            "Cluster_ID": row["cluster_id"],
            "Cluster_size": int(row["cluster_size"]),
            "Feature_raw": row["champion"],
            "Feature": feature_clean,
            "Champion_score": float(row["champion_score"]),
            "Importance_mean_SHAP": float(row["champion_importance_mean"]),
            "FreqTopK": float(row["champion_topk_freq_ratio"])
        })

champions_all = pd.DataFrame(records)

print("Champion records loaded:", len(champions_all))
print(champions_all.head())


# ============================================================
# 5. Build cluster-compression summary table under rho=0.90
# ============================================================

cluster_summary_each_fold = []

for (method, fold), g in champions_all.groupby(["Method", "Fold"]):
    sizes = g["Cluster_size"]

    cluster_summary_each_fold.append({
        "Imputation workflow": method,
        "Fold": fold,
        "Candidate features": TOTAL_FEATURES,
        "No. of clusters / champions": len(sizes),
        "Reduction ratio": len(sizes) / TOTAL_FEATURES,
        "Median cluster size": sizes.median(),
        "Maximum cluster size": sizes.max(),
        "Singleton clusters": int((sizes == 1).sum()),
        "Clusters with size <= 3": int((sizes <= 3).sum()),
    })

cluster_summary_each_fold = pd.DataFrame(cluster_summary_each_fold)


def mean_sd_text(x, digits=1):
    return f"{x.mean():.{digits}f} +/- {x.std(ddof=1):.{digits}f}"


def mean_sd_text3(x):
    return f"{x.mean():.3f} +/- {x.std(ddof=1):.3f}"


table_x_rows = []

for method, g in cluster_summary_each_fold.groupby("Imputation workflow"):
    table_x_rows.append({
        "Imputation workflow": method,
        "Candidate features": TOTAL_FEATURES,
        "No. of clusters / champions": mean_sd_text(g["No. of clusters / champions"], 1),
        "Reduction ratio": mean_sd_text3(g["Reduction ratio"]),
        "Median cluster size": f"{g['Median cluster size'].median():.1f}",
        "Maximum cluster size": mean_sd_text(g["Maximum cluster size"], 1),
        "Singleton clusters": mean_sd_text(g["Singleton clusters"], 1),
        "Clusters with size <= 3": mean_sd_text(g["Clusters with size <= 3"], 1),
    })

table_x = pd.DataFrame(table_x_rows)

outer_f1_map = {
    "global-mean": "0.896 +/- 0.030",
    "KNN": "0.904 +/- 0.023"
}

table_x["Outer-test Macro-F1 of champions"] = (
    table_x["Imputation workflow"].map(outer_f1_map)
)

order = ["global-mean", "KNN"]
table_x["order"] = table_x["Imputation workflow"].apply(
    lambda x: order.index(x) if x in order else 99
)
table_x = (
    table_x
    .sort_values("order")
    .drop(columns="order")
    .reset_index(drop=True)
)

print("\nCluster-compression summary table:")
print(table_x)


# ============================================================
# 6. Build top stable cluster-champion table
# ============================================================

stable_summary = (
    champions_all
    .groupby("Feature")
    .agg(
        Occurrence=("Feature", "size"),
        Mean_champion_score=("Champion_score", "mean"),
        SD_champion_score=("Champion_score", "std"),
        Mean_importance=("Importance_mean_SHAP", "mean"),
        Mean_FreqTopK=("FreqTopK", "mean"),
        Mean_cluster_size=("Cluster_size", "mean")
    )
    .reset_index()
)

stable_summary["Feature category"] = stable_summary["Feature"].apply(classify_feature)

stable_summary = stable_summary.sort_values(
    ["Occurrence", "Mean_champion_score"],
    ascending=[False, False]
).reset_index(drop=True)

top_n = 15
table_1 = stable_summary.head(top_n).copy()
table_1.insert(0, "Rank", np.arange(1, len(table_1) + 1))

table_1["Occurrence / 10"] = table_1["Occurrence"].astype(str) + "/10"
table_1["Champion score"] = table_1.apply(
    lambda r: f"{r['Mean_champion_score']:.3f} +/- {r['SD_champion_score']:.3f}",
    axis=1
)

table_1_for_paper = table_1[[
    "Rank",
    "Feature",
    "Feature category",
    "Occurrence / 10",
    "Champion score"
]].copy()

print("\nTop stable cluster-champion table:")
print(table_1_for_paper)


# ============================================================
# 7. Save tables
# ============================================================

table_out_path = OUT_DIR / "table_cluster_champion_summary_rho090.xlsx"

with pd.ExcelWriter(table_out_path, engine="openpyxl") as writer:
    table_x.to_excel(
        writer,
        sheet_name="cluster_summary",
        index=False
    )
    table_1_for_paper.to_excel(
        writer,
        sheet_name="top15_champions",
        index=False
    )
    stable_summary.to_excel(
        writer,
        sheet_name="all_stable_champions",
        index=False
    )
    champions_all.to_excel(
        writer,
        sheet_name="all_champion_records",
        index=False
    )
    cluster_summary_each_fold.to_excel(
        writer,
        sheet_name="cluster_summary_each_fold",
        index=False
    )

print("Tables saved to:", table_out_path)


# ============================================================
# 8. Cluster-size distribution
# ============================================================

def cluster_size_bin(x):
    if x == 1:
        return "1"
    elif x == 2:
        return "2"
    elif x == 3:
        return "3"
    elif 4 <= x <= 5:
        return "4-5"
    elif 6 <= x <= 10:
        return "6-10"
    elif 11 <= x <= 20:
        return "11-20"
    elif 21 <= x <= 50:
        return "21-50"
    else:
        return ">50"


bin_order = ["1", "2", "3", "4-5", "6-10", "11-20", "21-50", ">50"]

champions_all["Cluster_size_bin"] = champions_all["Cluster_size"].apply(cluster_size_bin)

bin_records = []

for (method, fold), g in champions_all.groupby(["Method", "Fold"]):
    counts = (
        g["Cluster_size_bin"]
        .value_counts()
        .reindex(bin_order)
        .fillna(0)
    )

    for b in bin_order:
        bin_records.append({
            "Method": method,
            "Fold": fold,
            "Cluster size bin": b,
            "Count": counts[b]
        })

bin_df = pd.DataFrame(bin_records)

bin_summary = (
    bin_df
    .groupby(["Method", "Cluster size bin"])
    .agg(
        Mean_count=("Count", "mean"),
        SD_count=("Count", "std")
    )
    .reset_index()
)

bin_summary["bin_order"] = bin_summary["Cluster size bin"].apply(
    lambda x: bin_order.index(x)
)
bin_summary = bin_summary.sort_values(["bin_order", "Method"])


# ============================================================
# 9. Global plotting parameters
# ============================================================

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"


# ============================================================
# 10. Standalone Fig. 8a
# ============================================================

fig, ax = plt.subplots(figsize=(9.5, 6.2))

x = np.arange(len(bin_order))
width = 0.35

methods = ["global-mean", "KNN"]
offsets = [-width / 2, width / 2]

for method, offset in zip(methods, offsets):
    sub = (
        bin_summary[bin_summary["Method"] == method]
        .set_index("Cluster size bin")
        .reindex(bin_order)
    )

    ax.bar(
        x + offset,
        sub["Mean_count"],
        width=width,
        yerr=sub["SD_count"],
        capsize=4,
        edgecolor="black",
        linewidth=1.0,
        label=method
    )

ax.set_xlabel("Cluster size", fontsize=16, fontweight="bold")
ax.set_ylabel("Number of clusters", fontsize=16, fontweight="bold")
ax.set_title(
    "Cluster-Size Distribution under |rho| >= 0.90",
    fontsize=18,
    fontweight="bold",
    pad=10
)

ax.set_xticks(x)
ax.set_xticklabels(bin_order, fontsize=13, fontweight="bold")

ax.tick_params(axis="y", labelsize=13, width=1.4)
for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.legend(frameon=False, fontsize=13)
ax.grid(axis="y", linestyle="--", alpha=0.4)

for spine in ax.spines.values():
    spine.set_linewidth(1.4)

plt.tight_layout()

fig8a_path = OUT_DIR / "fig8a_cluster_size_distribution_rho090.png"
plt.savefig(fig8a_path, dpi=300, bbox_inches="tight")
plt.show()

print("Fig. 8a saved to:", fig8a_path)


# ============================================================
# 11. Standalone Fig. 8b: top stable champions
#     The occurrence labels are placed to the right of error bars.
# ============================================================

plot_df = table_1.copy()
plot_df = plot_df.sort_values("Mean_champion_score", ascending=True)

fig, ax = plt.subplots(figsize=(9.5, 7.0))

bars = ax.barh(
    plot_df["Feature"],
    plot_df["Mean_champion_score"],
    xerr=plot_df["SD_champion_score"],
    capsize=4,
    edgecolor="black",
    linewidth=1.0
)

ax.set_xlabel("Mean champion score", fontsize=16, fontweight="bold")
ax.set_ylabel("Cluster champion feature", fontsize=16, fontweight="bold")
ax.set_title(
    "Top Stable Cluster Champions under |rho| >= 0.90",
    fontsize=18,
    fontweight="bold",
    pad=10
)

ax.tick_params(axis="x", labelsize=13, width=1.4)
ax.tick_params(axis="y", labelsize=13, width=1.4)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

max_right = float(
    (plot_df["Mean_champion_score"] + plot_df["SD_champion_score"].fillna(0)).max()
)
if max_right <= 0:
    max_right = 1.0

ax.set_xlim(0, max_right * 1.23)

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    mean_score = float(row["Mean_champion_score"])
    sd_score = (
        float(row["SD_champion_score"])
        if not pd.isna(row["SD_champion_score"])
        else 0.0
    )
    occurrence = row["Occurrence / 10"]

    text_x = mean_score + sd_score + max_right * 0.035

    ax.text(
        text_x,
        bar.get_y() + bar.get_height() / 2,
        occurrence,
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold"
    )

ax.grid(axis="x", linestyle="--", alpha=0.4)

for spine in ax.spines.values():
    spine.set_linewidth(1.4)

plt.tight_layout()

fig8b_path = OUT_DIR / "fig8b_stable_cluster_champions_rho090.png"
plt.savefig(fig8b_path, dpi=300, bbox_inches="tight")
plt.show()

print("Fig. 8b saved to:", fig8b_path)


# ============================================================
# 12. Combined Fig. 8
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6.8))

# ---------- Panel a ----------
ax = axes[0]

for method, offset in zip(methods, offsets):
    sub = (
        bin_summary[bin_summary["Method"] == method]
        .set_index("Cluster size bin")
        .reindex(bin_order)
    )

    ax.bar(
        x + offset,
        sub["Mean_count"],
        width=width,
        yerr=sub["SD_count"],
        capsize=4,
        edgecolor="black",
        linewidth=1.0,
        label=method
    )

ax.set_xlabel("Cluster size", fontsize=15, fontweight="bold")
ax.set_ylabel("Number of clusters", fontsize=15, fontweight="bold")
ax.set_title(
    "(a) Cluster-Size Distribution",
    fontsize=17,
    fontweight="bold",
    pad=10
)

ax.set_xticks(x)
ax.set_xticklabels(bin_order, fontsize=12, fontweight="bold")
ax.tick_params(axis="y", labelsize=12, width=1.4)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.legend(frameon=False, fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.4)

for spine in ax.spines.values():
    spine.set_linewidth(1.4)


# ---------- Panel b ----------
ax = axes[1]

plot_df = table_1.copy().sort_values("Mean_champion_score", ascending=True)

bars = ax.barh(
    plot_df["Feature"],
    plot_df["Mean_champion_score"],
    xerr=plot_df["SD_champion_score"],
    capsize=4,
    edgecolor="black",
    linewidth=1.0
)

ax.set_xlabel("Mean champion score", fontsize=15, fontweight="bold")
ax.set_ylabel("Cluster champion feature", fontsize=15, fontweight="bold")
ax.set_title(
    "(b) Stable Cluster Champions",
    fontsize=17,
    fontweight="bold",
    pad=10
)

ax.tick_params(axis="x", labelsize=12, width=1.4)
ax.tick_params(axis="y", labelsize=12, width=1.4)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

max_right = float(
    (plot_df["Mean_champion_score"] + plot_df["SD_champion_score"].fillna(0)).max()
)
if max_right <= 0:
    max_right = 1.0

ax.set_xlim(0, max_right * 1.23)

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    mean_score = float(row["Mean_champion_score"])
    sd_score = (
        float(row["SD_champion_score"])
        if not pd.isna(row["SD_champion_score"])
        else 0.0
    )
    occurrence = row["Occurrence / 10"]

    text_x = mean_score + sd_score + max_right * 0.035

    ax.text(
        text_x,
        bar.get_y() + bar.get_height() / 2,
        occurrence,
        va="center",
        ha="left",
        fontsize=11,
        fontweight="bold"
    )

ax.grid(axis="x", linestyle="--", alpha=0.4)

for spine in ax.spines.values():
    spine.set_linewidth(1.4)

plt.tight_layout()

fig8_combined_path = OUT_DIR / "fig8_combined_cluster_and_champions_rho090.png"
plt.savefig(fig8_combined_path, dpi=300, bbox_inches="tight")
plt.show()

print("Combined Fig. 8 saved to:", fig8_combined_path)

print("\nAll tasks completed. Output directory:")
print(OUT_DIR)